# 🎯 Train Mới Hoàn Toàn `YOLO11s` (`yolo11s.pt`) — Bộ Dataset v5 (Bổ Sung Nhiễu, Mờ, Răng Cưa Scan)

Notebook này khởi tạo **hoàn toàn mới** từ trọng số gốc **`yolo11s.pt` (YOLO11 Small)** và huấn luyện trên **Synthetic + Real 3-Regions Dataset v5 (1,502 mẫu)** đã bổ sung:
- Nhiễu răng cưa 1-bit scan độ phân giải thấp (`pixelated_scan` như `#44: 3 +0.005/0`, `#45: 1 +0.1/0`, `#6: 0.845 +0.005/0`)
- Nhòe mực in đậm làm 2 số `0` xếp chồng dính sát nhau (`ink_bleed_blur`)
- Mờ quang học + nhiễu hạt nền giấy scan (`gaussian_blur_noise`) & nhiễu nén JPEG
- Giữ đầy đủ toàn bộ các case cũ (Nominal dài, 2 số thập phân `1.00`, số đơn `1/3/4/5/6`, lệch trục X/Y, gạch ngang đậm ở giữa, góc nghiêng `R0.02/C0.02`)

Sau khi train xong, Notebook sẽ tự động xuất ra:
1. **`yolo_3regions_best.onnx`** (để nạp thẳng vào cổng **2. Cổng Model YOLO (.onnx)** trong Mini Tool).
2. **`best_3regions.pt`** (để lưu vào **Thư viện .PT**).

In [ ]:
# 1. Cài đặt thư viện Ultralytics YOLO và ONNX
!pip install -q ultralytics onnx onnxruntime


In [ ]:
# 2. Giải nén gói yolo_3regions_finetune_pack.zip (nếu chạy trên Google Colab / Kaggle)
import os
import zipfile
from pathlib import Path
import yaml

ZIP_NAME = 'yolo_3regions_finetune_pack.zip'
if Path(ZIP_NAME).is_file():
    import shutil
    if Path('synthetic_3regions_dataset').is_dir():
        shutil.rmtree('synthetic_3regions_dataset')
    print(f'Đang giải nén {ZIP_NAME}...')
    with zipfile.ZipFile(ZIP_NAME, 'r') as zf:
        zf.extractall('.')
    print('✔ Giải nén hoàn tất!')

ds_dir = Path('synthetic_3regions_dataset').resolve()
assert ds_dir.is_dir(), 'Không tìm thấy thư mục synthetic_3regions_dataset! Hãy upload yolo_3regions_finetune_pack.zip lên Colab.'

# Cập nhật đường dẫn tuyệt đối của data.yaml theo môi trường hiện tại
yaml_path = ds_dir / 'data.yaml'
data_cfg = {
    'path': ds_dir.as_posix(),
    'train': 'train/images',
    'val': 'val/images',
    'names': {0: 'nominal', 1: 'tol_up', 2: 'tol_lower'},
}
yaml_path.write_text(yaml.safe_dump(data_cfg, sort_keys=False), encoding='utf-8')
print('✔ Đã cấu hình data.yaml tại:', yaml_path)


In [ ]:
# 3. Train HOÀN TOÀN MỚI trên YOLO11s ('yolo11s.pt') — Không dùng lại weight best_3regions.pt cũ
# Tự động cài đặt ultralytics & onnx nếu chưa chạy Cell 1
import subprocess, sys
try:
    import ultralytics, onnx, onnxruntime
except ImportError:
    print('Đang cài đặt ultralytics, onnx, onnxruntime...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics', 'onnx', 'onnxruntime'])

import torch
from pathlib import Path
from ultralytics import YOLO

yaml_path = Path('synthetic_3regions_dataset/data.yaml').resolve()
device = 0 if torch.cuda.is_available() else 'cpu'
print('Thiết bị huấn luyện:', 'GPU CUDA' if device == 0 else 'CPU')

BASE_MODEL = 'yolo11s.pt'  # Khởi tạo model hoàn toàn mới từ YOLO11s gốc
print('✔ Khởi tạo model mới từ gốc:', BASE_MODEL)
model = YOLO(BASE_MODEL)
results = model.train(
    data=str(yaml_path),
    epochs=90,
    imgsz=320,          # Kích thước tối ưu cho ảnh crop Blue-Box CAD
    batch=16,
    device=device,
    project='runs_3regions',
    name='cad_bluebox_3regions_v5_noisy',
    exist_ok=True,
    pretrained=True,
    optimizer='AdamW',
    lr0=0.001,
    cos_lr=True,
    box=10.0,           # Tăng trọng số Box Loss để bắt tọa độ cực khít cả trên ảnh mờ/nhiễu/răng cưa
    cls=1.0,
    hsv_h=0.0,
    hsv_s=0.1,
    hsv_v=0.25,         # Tăng biến thiên độ sáng/tương phản cho bản vẽ scan mờ
    mosaic=0.0,         # Tắt ghép mảnh làm đứt số
    mixup=0.0,
    copy_paste=0.0,
    degrees=0.0,
    translate=0.05,
    scale=0.18,
    shear=0.0,
    perspective=0.0,
    fliplr=0.0,         # Tuyệt đối không lật ngang bản vẽ/chữ số
    flipud=0.0,         # Tuyệt đối không lật dọc
)


In [ ]:
# 4. Xuất model sang chuẩn ONNX (yolo_3regions_best.onnx) để nạp vào Mini Tool
import shutil, subprocess, sys
from pathlib import Path
try:
    import ultralytics, onnx, onnxruntime
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics', 'onnx', 'onnxruntime'])
from ultralytics import YOLO

# Tự động lấy đường dẫn best.pt từ kết quả train (hỗ trợ mọi phiên bản Ultralytics trên Colab)
candidates = []
if 'results' in globals() and hasattr(results, 'save_dir'):
    candidates.append(Path(results.save_dir) / 'weights' / 'best.pt')
if 'model' in globals() and hasattr(getattr(model, 'trainer', None), 'best'):
    candidates.append(Path(model.trainer.best))
candidates.append(Path('runs_3regions/cad_bluebox_3regions/weights/best.pt'))
candidates.append(Path('runs/detect/runs_3regions/cad_bluebox_3regions/weights/best.pt'))
candidates.extend(sorted(Path('.').rglob('**/cad_bluebox_3regions*/weights/best.pt'), key=lambda p: p.stat().st_mtime, reverse=True))

best_trained_pt = next((p for p in candidates if p and p.is_file()), None)
assert best_trained_pt is not None, 'Không tìm thấy best.pt! Hãy kiểm tra Cell 3 đã chạy xong chưa.'
print('✔ Tìm thấy model đã train tại:', best_trained_pt.resolve())

trained_model = YOLO(str(best_trained_pt))
exported_onnx = Path(trained_model.export(format='onnx', imgsz=320, simplify=False, opset=12))

out_pt = Path('best_3regions.pt')
out_onnx = Path('yolo_3regions_best.onnx')
shutil.copy2(best_trained_pt, out_pt)
shutil.copy2(exported_onnx, out_onnx)

print('====================================================')
print('✔ Đã xuất file .PT mới  :', out_pt.resolve())
print('✔ Đã xuất file .ONNX mới:', out_onnx.resolve())
print('====================================================')


In [ ]:
# 5. Tải file yolo_3regions_best.onnx và best_3regions.pt về máy (nếu chạy trên Google Colab)
try:
    from google.colab import files
    files.download('yolo_3regions_best.onnx')
    files.download('best_3regions.pt')
except Exception:
    print('Đang chạy local/Kaggle — lấy trực tiếp 2 file yolo_3regions_best.onnx và best_3regions.pt trong thư mục hiện tại.')
